In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token= "",
    instance="",
    overwrite=True
)

print("IBM Quantum account saved successfully.")

IBM Quantum account saved successfully.


In [4]:
service = QiskitRuntimeService()
print("Connected successfully.")
print(service.backends())

Connected successfully.
[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


In [5]:
for backend in service.backends():
    print(backend.name, "-", backend.status().status_msg, "- queue:", backend.status().pending_jobs)

ibm_fez - active - queue: 8
ibm_marrakesh - active - queue: 9
ibm_kingston - active - queue: 10


In [6]:
backend = service.least_busy(operational=True, simulator=False)
print("Selected backend:", backend.name)

Selected backend: ibm_fez


In [7]:
print("Backend name:", backend.name)
print("Number of qubits:", backend.num_qubits)
print("Basis gates:", backend.basis_gates)

Backend name: ibm_fez
Number of qubits: 156
Basis gates: ['cz', 'id', 'rz', 'sx', 'x']


In [8]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import numpy as np

# Same random setup as Level 1/2, but small round count for real hardware (queue-friendly)
N_BITS_HW = 20
rng = np.random.default_rng()

alice_bits = rng.integers(0, 2, N_BITS_HW)
alice_bases = rng.choice(['Z', 'X'], size=N_BITS_HW)
bob_bases = rng.choice(['Z', 'X'], size=N_BITS_HW)

def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 'X':
        qc.h(0)
    return qc

def measure_qubit(qc, basis):
    if basis == 'X':
        qc.h(0)
    qc.measure(0, 0)
    return qc

# Build all circuits
bb84_circuits = []
for i in range(N_BITS_HW):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    qc = measure_qubit(qc, bob_bases[i])
    bb84_circuits.append(qc)

# Transpile for the real backend
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
transpiled_circuits = pm.run(bb84_circuits)

print(f"Prepared and transpiled {len(transpiled_circuits)} BB84 circuits for {backend.name}")

Prepared and transpiled 20 BB84 circuits for ibm_fez


In [9]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)
job = sampler.run(transpiled_circuits, shots=1)

print("Job submitted. Job ID:", job.job_id())
print("Status:", job.status())

Job submitted. Job ID: dabt4mg09bds739sjth0
Status: QUEUED


In [10]:
# Check status periodically (or just wait and re-run this cell later)
print("Current status:", job.status())

# Once status shows DONE, get the results:
result = job.result()
print("Job completed. Results retrieved.")

Current status: QUEUED
Job completed. Results retrieved.


In [11]:
bob_results_hw = []

for i in range(N_BITS_HW):
    counts = result[i].data.c.get_counts()
    outcome = list(counts.keys())[0]   # single shot, so only one outcome
    bob_results_hw.append(int(outcome))

bob_results_hw = np.array(bob_results_hw)
print("Bob's hardware results:", bob_results_hw)

Bob's hardware results: [1 1 1 0 0 1 1 1 1 0 0 1 0 0 0 0 0 1 1 1]


In [12]:
# Sifting — compare Alice's and Bob's bases, keep matches
sift_mask = alice_bases == bob_bases
alice_sifted_hw = alice_bits[sift_mask]
bob_sifted_hw = bob_results_hw[sift_mask]

# QBER calculation
mismatches_hw = np.sum(alice_sifted_hw != bob_sifted_hw)
qber_hw = mismatches_hw / len(alice_sifted_hw) if len(alice_sifted_hw) > 0 else float('nan')

print("=" * 50)
print(f"BB84 - LEVEL 3 RESULT ({backend.name})")
print("=" * 50)
print(f"Total qubits sent   : {N_BITS_HW}")
print(f"Sifted key length   : {len(alice_sifted_hw)}")
print(f"QBER                : {qber_hw:.4f}")
print(f"Keys match          : {np.array_equal(alice_sifted_hw, bob_sifted_hw)}")

BB84 - LEVEL 3 RESULT (ibm_fez)
Total qubits sent   : 20
Sifted key length   : 9
QBER                : 0.0000
Keys match          : True


In [13]:
# LM05 setup (same style as BB84 above, small round count for real hardware)

bob_bits_lm05_hw = rng.integers(0, 2, N_BITS_HW)
bob_bases_lm05_hw = rng.choice(['Z', 'X'], size=N_BITS_HW)
alice_modes_hw = rng.choice(['CM', 'MM'], size=N_BITS_HW)
alice_cm_bases_hw = rng.choice(['Z', 'X'], size=N_BITS_HW)
alice_message_bits_hw = rng.integers(0, 2, N_BITS_HW)

def prepare_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 'X':
        qc.h(0)
    return qc

def alice_cm_measure(qc, basis):
    if basis == 'X':
        qc.h(0)
    qc.measure(0, 0)
    return qc

def alice_mm_encode(qc, message_bit):
    if message_bit == 1:
        qc.y(0)
    return qc

def bob_final_measure(qc, basis):
    if basis == 'X':
        qc.h(0)
    qc.measure(0, 0)
    return qc

lm05_circuits = []
for i in range(N_BITS_HW):
    qc = prepare_qubit(bob_bits_lm05_hw[i], bob_bases_lm05_hw[i])
    if alice_modes_hw[i] == 'CM':
        qc = alice_cm_measure(qc, alice_cm_bases_hw[i])
    else:
        qc = alice_mm_encode(qc, alice_message_bits_hw[i])
        qc = bob_final_measure(qc, bob_bases_lm05_hw[i])
    lm05_circuits.append(qc)

# Transpile for the same backend
transpiled_lm05 = pm.run(lm05_circuits)

print(f"Prepared and transpiled {len(transpiled_lm05)} LM05 circuits for {backend.name}")

Prepared and transpiled 20 LM05 circuits for ibm_fez


In [14]:
job_lm05 = sampler.run(transpiled_lm05, shots=1)

print("LM05 job submitted. Job ID:", job_lm05.job_id())
print("Status:", job_lm05.status())

LM05 job submitted. Job ID: dabt78de36ac739fu7q0
Status: RUNNING


In [15]:
# Check status periodically (or wait and re-run this cell later)
print("Current status:", job_lm05.status())

# Once status shows DONE, get the results:
result_lm05 = job_lm05.result()
print("LM05 job completed. Results retrieved.")

Current status: DONE
LM05 job completed. Results retrieved.


In [16]:
lm05_results_hw = []

for i in range(N_BITS_HW):
    counts = result_lm05[i].data.c.get_counts()
    outcome = list(counts.keys())[0]   # single shot, so only one outcome
    lm05_results_hw.append(int(outcome))

lm05_results_hw = np.array(lm05_results_hw)
print("LM05 hardware results:", lm05_results_hw)

LM05 hardware results: [0 1 0 1 0 0 0 1 0 0 0 1 0 0 1 1 1 0 1 0]


In [17]:
cm_results_hw = []
mm_bob_decoded_hw = []

for i in range(N_BITS_HW):
    if alice_modes_hw[i] == 'CM':
        cm_results_hw.append(lm05_results_hw[i])
    else:  # MM
        outcome = lm05_results_hw[i]
        decoded_bit = 0 if outcome == bob_bits_lm05_hw[i] else 1
        mm_bob_decoded_hw.append(decoded_bit)

cm_results_hw = np.array(cm_results_hw)
mm_bob_decoded_hw = np.array(mm_bob_decoded_hw)

print("CM results (hardware):", cm_results_hw)
print("MM decoded bits (hardware):", mm_bob_decoded_hw)

CM results (hardware): [1 0 0 1 0 1 0 1]
MM decoded bits (hardware): [1 1 0 0 0 1 0 0 1 1 0 0]


In [18]:
# --- Control Mode QBER check ---
cm_mask_hw = alice_modes_hw == 'CM'
cm_bob_bits_hw = bob_bits_lm05_hw[cm_mask_hw]
cm_bob_bases_hw = bob_bases_lm05_hw[cm_mask_hw]
cm_alice_bases_hw = alice_cm_bases_hw[cm_mask_hw]

basis_match_mask_hw = cm_alice_bases_hw == cm_bob_bases_hw
cm_checkable_bob_hw = cm_bob_bits_hw[basis_match_mask_hw]
cm_checkable_alice_hw = cm_results_hw[basis_match_mask_hw]

cm_mismatches_hw = np.sum(cm_checkable_alice_hw != cm_checkable_bob_hw)
qber_lm05_hw = cm_mismatches_hw / len(cm_checkable_alice_hw) if len(cm_checkable_alice_hw) > 0 else float('nan')

# --- Sifted key (Message Mode only) ---
mm_mask_hw = alice_modes_hw == 'MM'
alice_sifted_lm05_hw = alice_message_bits_hw[mm_mask_hw]
bob_sifted_lm05_hw = mm_bob_decoded_hw

print("=" * 50)
print(f"LM05 - LEVEL 3 RESULT ({backend.name})")
print("=" * 50)
print(f"Total qubits sent   : {N_BITS_HW}")
print(f"Sifted key length   : {len(alice_sifted_lm05_hw)}")
print(f"QBER                : {qber_lm05_hw:.4f}")
print(f"Keys match          : {np.array_equal(alice_sifted_lm05_hw, bob_sifted_lm05_hw)}")

LM05 - LEVEL 3 RESULT (ibm_fez)
Total qubits sent   : 20
Sifted key length   : 12
QBER                : 0.0000
Keys match          : True


In [19]:
print("=" * 65)
print(f"LEVEL 3 — REAL HARDWARE COMPARISON ({backend.name})")
print("=" * 65)
print(f"{'Metric':<25}{'BB84':<20}{'LM05':<20}")
print("-" * 65)
print(f"{'Raw qubits sent':<25}{N_BITS_HW:<20}{N_BITS_HW:<20}")
print(f"{'Sifted key length':<25}{len(alice_sifted_hw):<20}{len(alice_sifted_lm05_hw):<20}")
print(f"{'QBER (real hardware)':<25}{qber_hw:<20.4f}{qber_lm05_hw:<20.4f}")
print()
print("Compare against Level 2 noisy-simulator predictions:")
print(f"  BB84 predicted QBER range : ~0.029 - 0.039")
print(f"  LM05 predicted QBER range : ~0.048 - 0.071")

LEVEL 3 — REAL HARDWARE COMPARISON (ibm_fez)
Metric                   BB84                LM05                
-----------------------------------------------------------------
Raw qubits sent          20                  20                  
Sifted key length        9                   12                  
QBER (real hardware)     0.0000              0.0000              

Compare against Level 2 noisy-simulator predictions:
  BB84 predicted QBER range : ~0.029 - 0.039
  LM05 predicted QBER range : ~0.048 - 0.071


In [20]:
# Part 1: Build and transpile
N_BITS_HW = 200
rng = np.random.default_rng()

alice_bits = rng.integers(0, 2, N_BITS_HW)
alice_bases = rng.choice(['Z', 'X'], size=N_BITS_HW)
bob_bases = rng.choice(['Z', 'X'], size=N_BITS_HW)

bb84_circuits = []
for i in range(N_BITS_HW):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    qc = measure_qubit(qc, bob_bases[i])
    bb84_circuits.append(qc)

transpiled_circuits = pm.run(bb84_circuits)
print(f"Prepared and transpiled {len(transpiled_circuits)} BB84 circuits for {backend.name}")

# Part 2: Submit
job = sampler.run(transpiled_circuits, shots=1)
print("Job submitted. Job ID:", job.job_id())
print("Status:", job.status())

Prepared and transpiled 200 BB84 circuits for ibm_fez
Job submitted. Job ID: dabtai809bds739sk4rg
Status: QUEUED


In [21]:
print("Current status:", job.status())

result = job.result()
print("Job completed. Results retrieved.")

Current status: RUNNING
Job completed. Results retrieved.


In [22]:
bob_results_hw = []

for i in range(N_BITS_HW):
    counts = result[i].data.c.get_counts()
    outcome = list(counts.keys())[0]
    bob_results_hw.append(int(outcome))

bob_results_hw = np.array(bob_results_hw)
print("Bob's hardware results (first 20):", bob_results_hw[:20])

Bob's hardware results (first 20): [0 0 1 1 1 0 0 0 1 0 1 1 1 1 1 0 0 0 1 0]


In [23]:
sift_mask = alice_bases == bob_bases
alice_sifted_hw = alice_bits[sift_mask]
bob_sifted_hw = bob_results_hw[sift_mask]

mismatches_hw = np.sum(alice_sifted_hw != bob_sifted_hw)
qber_hw = mismatches_hw / len(alice_sifted_hw) if len(alice_sifted_hw) > 0 else float('nan')

print("=" * 50)
print(f"BB84 - LEVEL 3 RESULT ({backend.name}, N={N_BITS_HW})")
print("=" * 50)
print(f"Total qubits sent   : {N_BITS_HW}")
print(f"Sifted key length   : {len(alice_sifted_hw)}")
print(f"QBER                : {qber_hw:.4f}")
print(f"Keys match          : {np.array_equal(alice_sifted_hw, bob_sifted_hw)}")

BB84 - LEVEL 3 RESULT (ibm_fez, N=200)
Total qubits sent   : 200
Sifted key length   : 105
QBER                : 0.0190
Keys match          : False


In [24]:
# Part 1: Build and transpile
bob_bits_lm05_hw = rng.integers(0, 2, N_BITS_HW)
bob_bases_lm05_hw = rng.choice(['Z', 'X'], size=N_BITS_HW)
alice_modes_hw = rng.choice(['CM', 'MM'], size=N_BITS_HW)
alice_cm_bases_hw = rng.choice(['Z', 'X'], size=N_BITS_HW)
alice_message_bits_hw = rng.integers(0, 2, N_BITS_HW)

lm05_circuits = []
for i in range(N_BITS_HW):
    qc = prepare_qubit(bob_bits_lm05_hw[i], bob_bases_lm05_hw[i])
    if alice_modes_hw[i] == 'CM':
        qc = alice_cm_measure(qc, alice_cm_bases_hw[i])
    else:
        qc = alice_mm_encode(qc, alice_message_bits_hw[i])
        qc = bob_final_measure(qc, bob_bases_lm05_hw[i])
    lm05_circuits.append(qc)

transpiled_lm05 = pm.run(lm05_circuits)
print(f"Prepared and transpiled {len(transpiled_lm05)} LM05 circuits for {backend.name}")

# Part 2: Submit
job_lm05 = sampler.run(transpiled_lm05, shots=1)
print("LM05 job submitted. Job ID:", job_lm05.job_id())
print("Status:", job_lm05.status())

Prepared and transpiled 200 LM05 circuits for ibm_fez
LM05 job submitted. Job ID: dabtcf8c4p7c738khcs0
Status: RUNNING


In [25]:
print("Current status:", job_lm05.status())

result_lm05 = job_lm05.result()
print("LM05 job completed. Results retrieved.")

Current status: DONE
LM05 job completed. Results retrieved.


In [26]:
lm05_results_hw = []

for i in range(N_BITS_HW):
    counts = result_lm05[i].data.c.get_counts()
    outcome = list(counts.keys())[0]
    lm05_results_hw.append(int(outcome))

lm05_results_hw = np.array(lm05_results_hw)

cm_results_hw = []
mm_bob_decoded_hw = []

for i in range(N_BITS_HW):
    if alice_modes_hw[i] == 'CM':
        cm_results_hw.append(lm05_results_hw[i])
    else:  # MM
        outcome = lm05_results_hw[i]
        decoded_bit = 0 if outcome == bob_bits_lm05_hw[i] else 1
        mm_bob_decoded_hw.append(decoded_bit)

cm_results_hw = np.array(cm_results_hw)
mm_bob_decoded_hw = np.array(mm_bob_decoded_hw)

print("CM results (first 10):", cm_results_hw[:10])
print("MM decoded bits (first 10):", mm_bob_decoded_hw[:10])

CM results (first 10): [0 0 1 0 1 1 1 1 0 1]
MM decoded bits (first 10): [1 0 0 1 1 0 1 0 0 0]


In [27]:
cm_mask_hw = alice_modes_hw == 'CM'
cm_bob_bits_hw = bob_bits_lm05_hw[cm_mask_hw]
cm_bob_bases_hw = bob_bases_lm05_hw[cm_mask_hw]
cm_alice_bases_hw = alice_cm_bases_hw[cm_mask_hw]

basis_match_mask_hw = cm_alice_bases_hw == cm_bob_bases_hw
cm_checkable_bob_hw = cm_bob_bits_hw[basis_match_mask_hw]
cm_checkable_alice_hw = cm_results_hw[basis_match_mask_hw]

cm_mismatches_hw = np.sum(cm_checkable_alice_hw != cm_checkable_bob_hw)
qber_lm05_hw = cm_mismatches_hw / len(cm_checkable_alice_hw) if len(cm_checkable_alice_hw) > 0 else float('nan')

mm_mask_hw = alice_modes_hw == 'MM'
alice_sifted_lm05_hw = alice_message_bits_hw[mm_mask_hw]
bob_sifted_lm05_hw = mm_bob_decoded_hw

print("=" * 50)
print(f"LM05 - LEVEL 3 RESULT ({backend.name}, N={N_BITS_HW})")
print("=" * 50)
print(f"Total qubits sent   : {N_BITS_HW}")
print(f"Sifted key length   : {len(alice_sifted_lm05_hw)}")
print(f"QBER                : {qber_lm05_hw:.4f}")
print(f"Keys match          : {np.array_equal(alice_sifted_lm05_hw, bob_sifted_lm05_hw)}")

LM05 - LEVEL 3 RESULT (ibm_fez, N=200)
Total qubits sent   : 200
Sifted key length   : 91
QBER                : 0.0000
Keys match          : False


In [28]:
print("=" * 65)
print(f"LEVEL 3 — REAL HARDWARE COMPARISON ({backend.name}, N={N_BITS_HW})")
print("=" * 65)
print(f"{'Metric':<25}{'BB84':<20}{'LM05':<20}")
print("-" * 65)
print(f"{'Raw qubits sent':<25}{N_BITS_HW:<20}{N_BITS_HW:<20}")
print(f"{'Sifted key length':<25}{len(alice_sifted_hw):<20}{len(alice_sifted_lm05_hw):<20}")
print(f"{'QBER (real hardware)':<25}{qber_hw:<20.4f}{qber_lm05_hw:<20.4f}")
print()
print("Compare against Level 2 noisy-simulator predictions:")
print(f"  BB84 predicted QBER range : ~0.029 - 0.039")
print(f"  LM05 predicted QBER range : ~0.048 - 0.071")

LEVEL 3 — REAL HARDWARE COMPARISON (ibm_fez, N=200)
Metric                   BB84                LM05                
-----------------------------------------------------------------
Raw qubits sent          200                 200                 
Sifted key length        105                 91                  
QBER (real hardware)     0.0190              0.0000              

Compare against Level 2 noisy-simulator predictions:
  BB84 predicted QBER range : ~0.029 - 0.039
  LM05 predicted QBER range : ~0.048 - 0.071


In [29]:
print("LM05 checkable CM sample size:", len(cm_checkable_alice_hw))
print("BB84 sifted key length       :", len(alice_sifted_hw))

LM05 checkable CM sample size: 51
BB84 sifted key length       : 105


In [30]:
N_BITS_HW = 500
rng = np.random.default_rng()

alice_bits = rng.integers(0, 2, N_BITS_HW)
alice_bases = rng.choice(['Z', 'X'], size=N_BITS_HW)
bob_bases = rng.choice(['Z', 'X'], size=N_BITS_HW)

bb84_circuits = []
for i in range(N_BITS_HW):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    qc = measure_qubit(qc, bob_bases[i])
    bb84_circuits.append(qc)

transpiled_circuits = pm.run(bb84_circuits)
print(f"Prepared and transpiled {len(transpiled_circuits)} BB84 circuits for {backend.name}")

job = sampler.run(transpiled_circuits, shots=1)
print("Job submitted. Job ID:", job.job_id())
print("Status:", job.status())

Prepared and transpiled 500 BB84 circuits for ibm_fez
Job submitted. Job ID: dabtieo09bds739skfh0
Status: RUNNING


In [32]:
print("Status:", job.status())

Status: ERROR


In [33]:
job = sampler.run(transpiled_circuits, shots=1)
print("Job re-submitted. Job ID:", job.job_id())
print("Status:", job.status())

Job re-submitted. Job ID: dabtn3809bds739skl2g
Status: QUEUED


In [34]:
print("Current status:", job.status())

result = job.result()
print("Job completed. Results retrieved.")

Current status: DONE
Job completed. Results retrieved.


In [35]:
bob_results_hw = []
for i in range(N_BITS_HW):
    counts = result[i].data.c.get_counts()
    outcome = list(counts.keys())[0]
    bob_results_hw.append(int(outcome))
bob_results_hw = np.array(bob_results_hw)

sift_mask = alice_bases == bob_bases
alice_sifted_hw = alice_bits[sift_mask]
bob_sifted_hw = bob_results_hw[sift_mask]

mismatches_hw = np.sum(alice_sifted_hw != bob_sifted_hw)
qber_hw = mismatches_hw / len(alice_sifted_hw) if len(alice_sifted_hw) > 0 else float('nan')

print("=" * 50)
print(f"BB84 - LEVEL 3 RESULT ({backend.name}, N={N_BITS_HW})")
print("=" * 50)
print(f"Total qubits sent   : {N_BITS_HW}")
print(f"Sifted key length   : {len(alice_sifted_hw)}")
print(f"QBER                : {qber_hw:.4f}")
print(f"Keys match          : {np.array_equal(alice_sifted_hw, bob_sifted_hw)}")

BB84 - LEVEL 3 RESULT (ibm_fez, N=500)
Total qubits sent   : 500
Sifted key length   : 242
QBER                : 0.0041
Keys match          : False


In [36]:
# Part 1: Build and transpile
bob_bits_lm05_hw = rng.integers(0, 2, N_BITS_HW)
bob_bases_lm05_hw = rng.choice(['Z', 'X'], size=N_BITS_HW)
alice_modes_hw = rng.choice(['CM', 'MM'], size=N_BITS_HW)
alice_cm_bases_hw = rng.choice(['Z', 'X'], size=N_BITS_HW)
alice_message_bits_hw = rng.integers(0, 2, N_BITS_HW)

lm05_circuits = []
for i in range(N_BITS_HW):
    qc = prepare_qubit(bob_bits_lm05_hw[i], bob_bases_lm05_hw[i])
    if alice_modes_hw[i] == 'CM':
        qc = alice_cm_measure(qc, alice_cm_bases_hw[i])
    else:
        qc = alice_mm_encode(qc, alice_message_bits_hw[i])
        qc = bob_final_measure(qc, bob_bases_lm05_hw[i])
    lm05_circuits.append(qc)

transpiled_lm05 = pm.run(lm05_circuits)
print(f"Prepared and transpiled {len(transpiled_lm05)} LM05 circuits for {backend.name}")

# Part 2: Submit
job_lm05 = sampler.run(transpiled_lm05, shots=1)
print("LM05 job submitted. Job ID:", job_lm05.job_id())
print("Status:", job_lm05.status())

Prepared and transpiled 500 LM05 circuits for ibm_fez
LM05 job submitted. Job ID: dabto0te36ac739fut80
Status: QUEUED


In [37]:
print("Current status:", job_lm05.status())

result_lm05 = job_lm05.result()
print("LM05 job completed. Results retrieved.")

Current status: RUNNING
LM05 job completed. Results retrieved.


In [38]:
lm05_results_hw = []

for i in range(N_BITS_HW):
    counts = result_lm05[i].data.c.get_counts()
    outcome = list(counts.keys())[0]
    lm05_results_hw.append(int(outcome))

lm05_results_hw = np.array(lm05_results_hw)

cm_results_hw = []
mm_bob_decoded_hw = []

for i in range(N_BITS_HW):
    if alice_modes_hw[i] == 'CM':
        cm_results_hw.append(lm05_results_hw[i])
    else:  # MM
        outcome = lm05_results_hw[i]
        decoded_bit = 0 if outcome == bob_bits_lm05_hw[i] else 1
        mm_bob_decoded_hw.append(decoded_bit)

cm_results_hw = np.array(cm_results_hw)
mm_bob_decoded_hw = np.array(mm_bob_decoded_hw)

print("CM results (first 10):", cm_results_hw[:10])
print("MM decoded bits (first 10):", mm_bob_decoded_hw[:10])

CM results (first 10): [1 0 1 1 1 1 0 0 0 0]
MM decoded bits (first 10): [1 0 0 0 0 0 1 0 1 1]


In [39]:
cm_mask_hw = alice_modes_hw == 'CM'
cm_bob_bits_hw = bob_bits_lm05_hw[cm_mask_hw]
cm_bob_bases_hw = bob_bases_lm05_hw[cm_mask_hw]
cm_alice_bases_hw = alice_cm_bases_hw[cm_mask_hw]

basis_match_mask_hw = cm_alice_bases_hw == cm_bob_bases_hw
cm_checkable_bob_hw = cm_bob_bits_hw[basis_match_mask_hw]
cm_checkable_alice_hw = cm_results_hw[basis_match_mask_hw]

cm_mismatches_hw = np.sum(cm_checkable_alice_hw != cm_checkable_bob_hw)
qber_lm05_hw = cm_mismatches_hw / len(cm_checkable_alice_hw) if len(cm_checkable_alice_hw) > 0 else float('nan')

mm_mask_hw = alice_modes_hw == 'MM'
alice_sifted_lm05_hw = alice_message_bits_hw[mm_mask_hw]
bob_sifted_lm05_hw = mm_bob_decoded_hw

print("=" * 50)
print(f"LM05 - LEVEL 3 RESULT ({backend.name}, N={N_BITS_HW})")
print("=" * 50)
print(f"Total qubits sent   : {N_BITS_HW}")
print(f"Sifted key length   : {len(alice_sifted_lm05_hw)}")
print(f"QBER                : {qber_lm05_hw:.4f}")
print(f"Keys match          : {np.array_equal(alice_sifted_lm05_hw, bob_sifted_lm05_hw)}")
print(f"CM checkable sample size : {len(cm_checkable_alice_hw)}")

LM05 - LEVEL 3 RESULT (ibm_fez, N=500)
Total qubits sent   : 500
Sifted key length   : 247
QBER                : 0.0079
Keys match          : False
CM checkable sample size : 127


In [40]:
print("=" * 65)
print(f"LEVEL 3 — REAL HARDWARE COMPARISON ({backend.name}, N={N_BITS_HW})")
print("=" * 65)
print(f"{'Metric':<25}{'BB84':<20}{'LM05':<20}")
print("-" * 65)
print(f"{'Raw qubits sent':<25}{N_BITS_HW:<20}{N_BITS_HW:<20}")
print(f"{'Sifted key length':<25}{len(alice_sifted_hw):<20}{len(alice_sifted_lm05_hw):<20}")
print(f"{'QBER (real hardware)':<25}{qber_hw:<20.4f}{qber_lm05_hw:<20.4f}")
print(f"{'Checkable sample size':<25}{len(alice_sifted_hw):<20}{len(cm_checkable_alice_hw):<20}")

LEVEL 3 — REAL HARDWARE COMPARISON (ibm_fez, N=500)
Metric                   BB84                LM05                
-----------------------------------------------------------------
Raw qubits sent          500                 500                 
Sifted key length        242                 247                 
QBER (real hardware)     0.0041              0.0079              
Checkable sample size    242                 127                 
